# 05 - Scaling and DataLoaders

## Goal
- prepare scaled feature matrices for train/val/test,
- keep strict anti-leakage rule (`fit` scaler only on train),
- compute class imbalance handling setup,
- build PyTorch `Dataset`/`DataLoader`,
- save preprocessing artifacts for model training notebooks.


In [1]:
import json
import os
import pickle

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.preprocessing import RobustScaler, StandardScaler
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler


In [2]:
try:
    from stylin import InfoDisplayStyler
    styler = InfoDisplayStyler()
except Exception:
    class _FallbackStyler:
        def style_me(self, obj, title=None):
            if title:
                print(f"\n=== {title} ===")
            display(obj)

        def show_line(self, *args, sep=" ", title=None):
            text = sep.join(str(a) for a in args).strip()
            if title:
                print(f"{title}: {text}")
            else:
                print(text)

        def show_meta(self, data):
            print("shape:", getattr(data, "shape", None))
            display(data.head(3) if hasattr(data, "head") else data)

    styler = _FallbackStyler()


In [3]:
DATA_PATH = '../../data/raw/ethusdt_1h.csv'
CLEAN_PATH = '../../data/clean/'
LABELED_PATH = '../../data/labeled/'
PROCESSED_PATH = '../../data/processed/'
CHECKPOINTS_PATH = '../../checkpoints/'

TRAIN_SELECTED_FILE = PROCESSED_PATH + 'train_selected.csv'
VAL_SELECTED_FILE = PROCESSED_PATH + 'val_selected.csv'
TEST_SELECTED_FILE = PROCESSED_PATH + 'test_selected.csv'
SELECTED_FEATURES_FILE = PROCESSED_PATH + 'selected_feature_columns.csv'

assert os.path.exists(TRAIN_SELECTED_FILE), 'Missing train_selected.csv. Run notebook 04 first.'
assert os.path.exists(VAL_SELECTED_FILE), 'Missing val_selected.csv. Run notebook 04 first.'
assert os.path.exists(TEST_SELECTED_FILE), 'Missing test_selected.csv. Run notebook 04 first.'
assert os.path.exists(SELECTED_FEATURES_FILE), 'Missing selected_feature_columns.csv. Run notebook 04 first.'

os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(CHECKPOINTS_PATH, exist_ok=True)

train_df = pd.read_csv(TRAIN_SELECTED_FILE)
val_df = pd.read_csv(VAL_SELECTED_FILE)
test_df = pd.read_csv(TEST_SELECTED_FILE)
selected_features = pd.read_csv(SELECTED_FEATURES_FILE)['feature_column'].tolist()

styler.show_meta(train_df)
styler.style_me(train_df.head(5), title='Train selected preview')


,timestamp,target,future_return,open,quote_asset_volume,number_of_trades,return_1,return_12,return_24,lower_shadow,body_size,rolling_return_std_6,rolling_return_skew_6,rolling_return_kurt_6,rolling_return_skew_12,rolling_return_kurt_12,rolling_return_skew_24,rolling_return_kurt_24,rolling_return_skew_48,rolling_return_kurt_48,momentum_6,macd,macd_hist,bb_width,bb_position,volume_return,volume_std_24,volume_zscore_24,hour_sin,hour_cos,dow_sin,dow_cos,is_weekend,return_1_lag_1,return_1_lag_2,return_1_lag_3,return_6_lag_1,return_6_lag_3,volume_zscore_24_lag_1,volume_zscore_24_lag_2,volume_zscore_24_lag_3
0,2021-01-03 00:00:00+00:00,2,0.031129,774.440000,26814528.488316,28322,0.001614,0.031141,0.077664,0.004963,0.001766,0.018041,-1.948287,4.513416,-1.259378,2.632827,-0.643982,3.065160,-0.136671,3.567542,-4.200000,10.765491,0.922889,0.110961,0.719223,-0.096854,43008.063166,-0.495087,0.000000,1.000000,-0.781831,0.623490,1,0.003069,0.004214,0.016390,-0.005495,0.000677,-0.421914,-0.774749,0.337353
1,2021-01-03 01:00:00+00:00,2,0.029358,775.840000,21663976.812176,18434,0.001495,0.008044,0.065247,0.004492,0.001454,0.017784,-1.855123,4.341083,-1.630953,3.772491,-0.568431,3.282952,-0.162931,4.256054,-7.820000,10.695492,0.682312,0.108214,0.710874,-0.192820,43238.714742,-0.636960,0.258819,0.965926,-0.781831,0.623490,1,0.001614,0.003069,0.004214,-0.005385,-0.012469,-0.495087,-0.421914,-0.774749
2,2021-01-03 02:00:00+00:00,2,0.049837,776.980000,18364302.756458,16621,0.002535,0.023655,0.067860,0.001926,0.002516,0.005724,2.285897,5.357582,-2.109688,6.289927,-0.595568,3.330025,-0.211759,4.391384,22.390000,10.675915,0.530188,0.106388,0.712361,-0.154887,43181.994689,-0.739582,0.500000,0.866025,-0.781831,0.623490,1,0.001495,0.001614,0.003069,-0.009964,-0.005495,-0.636960,-0.495087,-0.421914
3,2021-01-03 03:00:00+00:00,2,0.059661,778.950000,21882218.612996,20077,-0.008435,0.005127,0.057476,0.001165,0.008519,0.004609,-2.186385,5.066903,-1.742787,4.859522,-0.497115,2.873336,-0.163267,4.124836,3.420000,10.014811,-0.104732,0.101508,0.613236,0.196276,42644.678121,-0.655058,0.707107,0.707107,-0.781831,0.623490,1,0.002535,0.001495,0.001614,0.029595,-0.005385,-0.739582,-0.636960,-0.495087
4,2021-01-03 04:00:00+00:00,2,0.062142,772.440000,26595385.984909,25749,0.006525,-0.005793,0.065748,0.004798,0.006393,0.005034,-1.668141,3.830248,-2.141599,6.264532,-0.575808,2.934718,-0.220493,4.104552,5.220000,9.784775,-0.267815,0.095456,0.659218,0.219320,41925.785014,-0.539822,0.866025,0.500000,-0.781831,0.623490,1,-0.008435,0.002535,0.001495,0.004448,-0.009964,-0.655058,-0.739582,-0.636960


## 1. Input consistency checks


In [4]:
for df in [train_df, val_df, test_df]:
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')

required_cols = ['timestamp', 'target', 'future_return']
for name, part in [('train', train_df), ('val', val_df), ('test', test_df)]:
    missing = [c for c in required_cols if c not in part.columns]
    assert not missing, f'{name}: missing required columns {missing}'
    assert part['timestamp'].notna().all(), f'{name}: invalid timestamp values'

assert set(selected_features).issubset(set(train_df.columns)), 'Some selected features missing in train_df'
assert set(selected_features).issubset(set(val_df.columns)), 'Some selected features missing in val_df'
assert set(selected_features).issubset(set(test_df.columns)), 'Some selected features missing in test_df'

consistency_report = pd.DataFrame({
    'split': ['train', 'val', 'test'],
    'rows': [len(train_df), len(val_df), len(test_df)],
    'feature_count': [len(selected_features), len(selected_features), len(selected_features)],
    'nan_in_features': [
        int(train_df[selected_features].isna().sum().sum()),
        int(val_df[selected_features].isna().sum().sum()),
        int(test_df[selected_features].isna().sum().sum()),
    ],
})
styler.style_me(consistency_report, title='Input consistency report')


,split,rows,feature_count,nan_in_features
0,train,32326,38,0
1,val,6937,38,0
2,test,6939,38,0


## 2. Build X/y and choose scaler


In [5]:
X_train = train_df[selected_features].to_numpy(dtype=np.float32)
X_val = val_df[selected_features].to_numpy(dtype=np.float32)
X_test = test_df[selected_features].to_numpy(dtype=np.float32)

y_train = train_df['target'].to_numpy(dtype=np.int64)
y_val = val_df['target'].to_numpy(dtype=np.int64)
y_test = test_df['target'].to_numpy(dtype=np.int64)

assert np.isfinite(X_train).all(), 'X_train contains NaN/inf'
assert np.isfinite(X_val).all(), 'X_val contains NaN/inf'
assert np.isfinite(X_test).all(), 'X_test contains NaN/inf'


def estimate_outlier_ratio_iqr(X: np.ndarray) -> float:
    q1 = np.percentile(X, 25, axis=0)
    q3 = np.percentile(X, 75, axis=0)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    mask = (X < low) | (X > high)
    return float(mask.mean())

outlier_ratio = estimate_outlier_ratio_iqr(X_train)
AUTO_SCALER = 'robust' if outlier_ratio > 0.05 else 'standard'

SCALER_TYPE = AUTO_SCALER  # override manually if needed: 'standard' / 'robust'
if SCALER_TYPE == 'standard':
    scaler = StandardScaler()
elif SCALER_TYPE == 'robust':
    scaler = RobustScaler()
else:
    raise ValueError(f'Unsupported SCALER_TYPE: {SCALER_TYPE}')

styler.style_me(
    pd.DataFrame({
        'metric': ['train_outlier_ratio_iqr', 'auto_scaler_choice', 'final_scaler_choice'],
        'value': [outlier_ratio, AUTO_SCALER, SCALER_TYPE],
    }),
    title='Scaler choice rationale',
)


,metric,value
0,train_outlier_ratio_iqr,0.055609
1,auto_scaler_choice,robust
2,final_scaler_choice,robust


## 3. Anti-leakage scaling


In [6]:
# Fit scaler only on train, then transform val/test.
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

scale_report = pd.DataFrame({
    'split': ['train_scaled', 'val_scaled', 'test_scaled'],
    'rows': [len(X_train_scaled), len(X_val_scaled), len(X_test_scaled)],
    'cols': [X_train_scaled.shape[1], X_val_scaled.shape[1], X_test_scaled.shape[1]],
    'mean_abs_sample': [
        float(np.abs(X_train_scaled[:, : min(10, X_train_scaled.shape[1])]).mean()),
        float(np.abs(X_val_scaled[:, : min(10, X_val_scaled.shape[1])]).mean()),
        float(np.abs(X_test_scaled[:, : min(10, X_test_scaled.shape[1])]).mean()),
    ],
})
styler.style_me(scale_report, title='Scaling report')


,split,rows,cols,mean_abs_sample
0,train_scaled,32326,38,0.704770
1,val_scaled,6937,38,0.970619
2,test_scaled,6939,38,1.157225


## 4. Class imbalance setup


In [7]:
class_counts = np.bincount(y_train)
num_classes = int(class_counts.shape[0])

# Balanced class weights: N / (K * n_c)
class_weights = len(y_train) / (num_classes * np.maximum(class_counts, 1))
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

class_report = pd.DataFrame({
    'class_id': list(range(num_classes)),
    'count_train': class_counts,
    'class_weight': class_weights,
})
styler.style_me(class_report, title='Train class distribution and weights')


,class_id,count_train,class_weight
0,0,8153,1.321640
1,1,15327,0.703030
2,2,8846,1.218102


## 5. Dataset and DataLoader


In [8]:
class TabularDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 128
NUM_WORKERS = 0
USE_WEIGHTED_SAMPLER = False  # True if you want sampler instead of class weights

train_dataset = TabularDataset(X_train_scaled, y_train)
val_dataset = TabularDataset(X_val_scaled, y_val)
test_dataset = TabularDataset(X_test_scaled, y_test)

if USE_WEIGHTED_SAMPLER:
    sample_weights = class_weights[y_train]
    sampler = WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.float32),
        num_samples=len(sample_weights),
        replacement=True,
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=NUM_WORKERS,
    )
else:
    sampler = None
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
    )

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

loader_report = pd.DataFrame({
    'loader': ['train', 'val', 'test'],
    'dataset_size': [len(train_dataset), len(val_dataset), len(test_dataset)],
    'batch_size': [BATCH_SIZE, BATCH_SIZE, BATCH_SIZE],
    'num_batches': [len(train_loader), len(val_loader), len(test_loader)],
    'sampler': [
        'WeightedRandomSampler' if USE_WEIGHTED_SAMPLER else 'shuffle',
        'sequential',
        'sequential',
    ],
})
styler.style_me(loader_report, title='DataLoader report')


,loader,dataset_size,batch_size,num_batches,sampler
0,train,32326,128,253,shuffle
1,val,6937,128,55,sequential
2,test,6939,128,55,sequential


## 6. Sanity check batches


In [9]:
x_batch, y_batch = next(iter(train_loader))

batch_report = pd.DataFrame({
    'metric': ['x_batch_shape', 'y_batch_shape', 'x_dtype', 'y_dtype'],
    'value': [str(tuple(x_batch.shape)), str(tuple(y_batch.shape)), str(x_batch.dtype), str(y_batch.dtype)],
})
styler.style_me(batch_report, title='Batch sanity check')


,metric,value
0,x_batch_shape,"(128, 38)"
1,y_batch_shape,"(128,)"
2,x_dtype,torch.float32
3,y_dtype,torch.int64


## 7. Save preprocessing artifacts


In [10]:
SCALER_OUT = PROCESSED_PATH + 'selected_scaler.pkl'
CLASS_WEIGHTS_OUT = PROCESSED_PATH + 'class_weights.csv'
PREP_SUMMARY_OUT = PROCESSED_PATH + 'scaling_dataloaders_summary.json'
ARRAYS_OUT = PROCESSED_PATH + 'scaled_arrays.npz'

with open(SCALER_OUT, 'wb') as f:
    pickle.dump(scaler, f)

pd.DataFrame({'class_id': list(range(num_classes)), 'class_weight': class_weights}).to_csv(CLASS_WEIGHTS_OUT, index=False)

np.savez_compressed(
    ARRAYS_OUT,
    X_train_scaled=X_train_scaled,
    X_val_scaled=X_val_scaled,
    X_test_scaled=X_test_scaled,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
)

summary = {
    'scaler_type': SCALER_TYPE,
    'auto_scaler': AUTO_SCALER,
    'train_outlier_ratio_iqr': float(outlier_ratio),
    'selected_feature_count': int(len(selected_features)),
    'train_shape': [int(X_train_scaled.shape[0]), int(X_train_scaled.shape[1])],
    'val_shape': [int(X_val_scaled.shape[0]), int(X_val_scaled.shape[1])],
    'test_shape': [int(X_test_scaled.shape[0]), int(X_test_scaled.shape[1])],
    'batch_size': int(BATCH_SIZE),
    'use_weighted_sampler': bool(USE_WEIGHTED_SAMPLER),
    'class_weights': {str(i): float(w) for i, w in enumerate(class_weights)},
}

with open(PREP_SUMMARY_OUT, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

styler.show_line(SCALER_OUT, title='Saved scaler')
styler.show_line(CLASS_WEIGHTS_OUT, title='Saved class weights')
styler.show_line(ARRAYS_OUT, title='Saved scaled arrays')
styler.show_line(PREP_SUMMARY_OUT, title='Saved preprocessing summary')


## 8. Final conclusions (fill after run)
- Which scaler was selected and why?
- Was anti-leakage scaling correctly applied (fit on train only)?
- What class imbalance strategy will you use in training (`class_weights` vs `sampler`)?
- Is data ready for notebook 06 (MLP + train/validate/fit)?

Wnioski końcowe (Notebook 05)
W notebooku poprawnie przygotowano dane wejściowe do treningu modelu MLP, zachowując zasadę anty-leakage: skaler został dopasowany wyłącznie na zbiorze treningowym, a następnie zastosowany do walidacji i testu.
Po wcześniejszej poprawce inf -> NaN -> dropna dane wejściowe nie zawierają już wartości NaN ani inf w cechach modelowych.

Finalne rozmiary macierzy po preprocessingu:

train: 32 326 x 38
val: 6 937 x 38
test: 6 939 x 38
Automatyczna diagnostyka outlierów (train_outlier_ratio_iqr ≈ 0.0556) wskazała na wybór RobustScaler, co jest uzasadnione dla danych finansowych z ogonami rozkładów i skokami zmienności.

Dla niezbalansowanych klas obliczono wagi (na train):

klasa 0: 1.3216
klasa 1: 0.7030
klasa 2: 1.2181
W konfiguracji notebooka zastosowano podejście z wagami klas (use_weighted_sampler = False), co jest spójne z użyciem CrossEntropyLoss(weight=...) w kolejnych etapach.

Artefakty preprocessingu zostały zapisane poprawnie (selected_scaler.pkl, class_weights.csv, scaled_arrays.npz, scaling_dataloaders_summary.json), a dane są gotowe do notebooka 06 (architektura MLP + trening/walidacja).

